# SPK-3 — Driver OOM (collecting unbounded data)

**Break → Detect → Fix → Prove.** The driver is a **single JVM**. Pulling a generated-large
frame back to it with `.collect()` / `.toPandas()` (or broadcasting something huge) blows its
heap — more executors can't help, because the result funnels through the one driver process.
The lesson in one line: **never collect unbounded data to the driver.**

**Pre-requisite:** the unified Spark server is up (`make up`). This notebook connects via Spark
Connect. **Open the Spark UI at http://localhost:4040** and watch it while the cells run.

**Laptop-safe — the failure is *contained*.** `conf/spark-defaults.conf` sets
`spark.driver.maxResultSize 1g`, so an oversized `.collect()` raises a **clean exception**
(Spark refuses to ship a result bigger than the cap) instead of letting the driver heap die and
freezing the machine. We catch it in a `try/except` so the notebook keeps running. Data is
generated lazily and nothing is written, so the default **tuned** box is fine (no
`make up-constrained`). Nothing to delete at the end.

See the companion writeup in [`README.md`](./README.md), the
[Spark-UI guide](../../docs/spark-ui-guide.md), and the [troubleshooting sheet](../../docs/troubleshooting.md).

In [1]:
from common.spark_session import spark, display_df
from common.profiles import apply_profile, profile_summary
from common.datagen import uniform_keys, wide_rows, key_dimension
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

# Watch this while the cells run — the collect's job and the driver-side stall show up here.
print("Spark UI:", "http://localhost:4040")
spark

Spark UI: http://localhost:4040


## Step 0 — Parameters & the generated-large frame

We synthesize a **wide** fact (`wide_rows`: ~50 doubles per row, so each row is many bytes
*serialized*) at tens of millions of rows — all lazily, nothing stored. Its full materialized
size is far bigger than the 1 GB `spark.driver.maxResultSize` cap, which is exactly what makes a
naive `.collect()` fail cleanly. A `limit(3)` peek is safe (only 3 rows cross to the driver).

In [4]:
# Lower N_ROWS if your laptop is slow; the point is only that the FULL result exceeds 1 GB.
N_ROWS  = 15_000_000
N_COLS  = 50          # wide rows -> large bytes-per-row -> result blows past maxResultSize fast
N_KEYS  = 2_000       # for the aggregation fix + the small dimension

wide_fact = wide_rows(spark, n_rows=N_ROWS, n_cols=N_COLS)          # the "too big to collect" frame
fact      = uniform_keys(spark, n_rows=N_ROWS, n_keys=N_KEYS,
                         key_col="customer_id")                     # for the cluster-side aggregation
dim       = key_dimension(spark, n_keys=N_KEYS, key_col="customer_id")  # genuinely small -> safe to broadcast

print(f"wide_fact: ~{N_ROWS:,} rows x {N_COLS} cols (lazy)   dim: {N_KEYS:,} rows")
print(f"driver.maxResultSize = {spark.conf.get('spark.driver.maxResultSize', '<unset>')}"
      f"   driver.memory = {spark.conf.get('spark.driver.memory', '<unset>')}")
wide_fact.limit(3).toPandas()                                       # safe: only 3 rows cross to the driver

wide_fact: ~15,000,000 rows x 50 cols (lazy)   dim: 2,000 rows
driver.maxResultSize = 1g   driver.memory = 2g


,row_id,c0,c1,c2,c3,c4,c5,c6,c7,c8,...,c40,c41,c42,c43,c44,c45,c46,c47,c48,c49
0,0,3.270820,81.491673,16.393709,38.313210,44.743318,6.422547,57.012845,1.001352,6.206111,...,68.695047,33.096191,38.489824,42.294009,79.556867,28.747133,36.796300,71.357072,67.897024,62.458317
1,1,30.928075,35.428439,77.098252,46.738159,90.257122,40.353511,51.433143,51.869064,20.517604,...,19.149504,72.723184,79.946629,29.509464,77.067809,35.786903,52.899815,40.525171,22.257572,90.219997
2,2,12.566721,14.921668,94.071408,82.960716,94.156990,25.106212,77.922221,92.814166,66.189267,...,45.239954,3.764571,44.262111,88.954163,17.196092,31.466402,76.212270,99.734881,19.062410,5.175960


## 1. Break it (A) — `.collect()` the whole wide frame

The classic mistake: finish a tens-of-millions-row frame the way you'd finish a 100-row sample.
Every partition's rows stream back to the **single driver JVM** to be assembled into one local
object — and the total blows past `spark.driver.maxResultSize` (1 GB). We wrap it in
`try/except` so the contained failure is printed and the notebook keeps going.

**Watch the Spark UI now:** the job's tasks finish on the cluster, then the driver fails pulling
the results back.

In [8]:
# A contained driver-OOM: Spark aborts because the serialized result exceeds maxResultSize.
def collect_everything():
    return wide_fact.collect()          # <-- the antipattern: unbounded collect to the driver

err_msg = None
try:
    m_broken = measure(spark, "collect (huge, fails)", collect_everything)
except Exception as exc:                # noqa: BLE001 — we EXPECT this to fail; keep the notebook alive
    err_msg = str(exc).splitlines()[:5]
    m_broken = {"label": "collect (huge, fails)", "runtime_s": None}
    print("CONTAINED FAILURE (this is the lesson):\n  ", err_msg)

print("\nNotebook is still running — the cap turned a driver-heap death into a clean exception.")

CONTAINED FAILURE (this is the lesson):
   ['(java.lang.IllegalStateException) Cannot call methods on a stopped SparkContext.', 'This stopped SparkContext was created at:', '', 'org.apache.spark.sql.hive.thriftserver.HiveThriftServer2.main(HiveThriftServer2.scala)', 'java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)']

Notebook is still running — the cap turned a driver-heap death into a clean exception.


## 1. Break it (B) — oversized broadcast (same root cause)

`F.broadcast(df)` asks Spark to **collect the whole side to the driver** and ship it to every
task. Broadcasting the *large* frame is a driver-OOM in disguise — the big side is funneled
through the driver first, tripping the same `maxResultSize` guardrail.

In [6]:
# Broadcasting a LARGE frame -> Spark must materialize it on the driver -> contained failure.
bad_broadcast = dim.join(F.broadcast(wide_fact.select("row_id")),
                         on=wide_fact["row_id"] == dim["customer_id"], how="inner")

try:
    bad_broadcast.count()               # triggers the broadcast of the huge side
    print("(broadcast unexpectedly succeeded — try raising N_ROWS)")
except Exception as exc:                # noqa: BLE001 — same family of driver-pressure failure
    print("CONTAINED FAILURE (oversized broadcast):\n  ", str(exc).splitlines()[:5])

CONTAINED FAILURE (oversized broadcast):
   ['Not enough memory to build and broadcast the table to all worker nodes. As a workaround, you can either disable broadcast by setting spark.sql.autoBroadcastJoinThreshold to -1 or increase the spark driver memory by setting spark.driver.memory to a higher value.', '', 'JVM stacktrace:', 'org.apache.spark.SparkException', '\tat org.apache.spark.sql.errors.QueryExecutionErrors$.notEnoughMemoryToBuildAndBroadcastTableError(QueryExecutionErrors.scala:2076)']


## 2. Detect it — read the Spark UI

http://localhost:4040 (see [`docs/spark-ui-guide.md`](../../docs/spark-ui-guide.md)):

- **Driver-side error message** — `Total size of serialized results ... is bigger than
  spark.driver.maxResultSize` is the definitive tell (without the cap it would be a driver
  `OutOfMemoryError`).
- **Jobs → Event Timeline** — the collect's tasks complete on the cluster, then a long
  **driver-side gap / failure** as results are pulled back — not cluster compute.
- **SQL / DataFrame** — the `Collect` action's query; the scan reads ~all rows.
- **Executors** — the **driver** row buffers the result; executors are idle by then.
- **Environment** — confirm `spark.driver.maxResultSize` and `spark.driver.memory`: the box the
  result must fit into.

The cell below echoes the captured error and the relevant confs so the signal is in the notebook too.

In [2]:
profile_summary(spark, keys=["spark.driver.maxResultSize", "spark.driver.memory",
                              "spark.sql.autoBroadcastJoinThreshold"])
print()
if err_msg:
    print("driver-side error captured from the broken collect:\n  ", err_msg)
else:
    print("No error captured — if the collect succeeded, raise N_ROWS / N_COLS so the "
          "result exceeds 1 GB.")

NumberFormatException: Size must be specified as bytes (b), kibibytes (k), mebibytes (m), gibibytes (g), tebibytes (t), or pebibytes(p). E.g. 50b, 100k, or 250m.
Failed to parse byte string: <unset>

JVM stacktrace:
java.lang.NumberFormatException
	at org.apache.spark.network.util.JavaUtils.byteStringAs(JavaUtils.java:332)
	at org.apache.spark.internal.config.ConfigHelpers$.byteFromString(ConfigBuilder.scala:68)
	at org.apache.spark.internal.config.ConfigBuilder.$anonfun$bytesConf$1(ConfigBuilder.scala:281)
	at org.apache.spark.internal.config.ConfigBuilder.$anonfun$bytesConf$1$adapted(ConfigBuilder.scala:281)
	at org.apache.spark.sql.internal.SQLConf.$anonfun$getConfString$4(SQLConf.scala:6686)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.internal.SQLConf.getConfString(SQLConf.scala:6681)
	at org.apache.spark.sql.classic.RuntimeConfig.get(RuntimeConfig.scala:59)
	at org.apache.spark.sql.connect.service.SparkConnectConfigHandler.$anonfun$handleGetWithDefault$1(

## 3. Fix it (A) — aggregate on the cluster, return a tiny result

Do the heavy work **distributed** and bring back only the summary. The `groupBy().agg()` shuffles
and aggregates across executors; only the small grouped result (a handful of rows per key) returns
to the driver — safe to `.toPandas()`.

In [5]:
agg = (fact.groupBy("customer_id")
            .agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue")))

# Aggregation runs on the cluster; only ~N_KEYS small rows come back to the driver.
m_agg = measure(spark, "aggregate->collect", lambda: agg.count())
print("aggregate metrics:", m_agg)
agg.orderBy(F.desc("revenue")).limit(5).toPandas()   # tiny, safe to pull to the driver

aggregate metrics: {'label': 'aggregate->collect', 'runtime_s': 0.44, 'num_tasks': 10, 'shuffle_read_bytes': 104147, 'shuffle_write_bytes': 104147, 'spill_mem_bytes': 0, 'spill_disk_bytes': 0, 'task_time_median_ms': 22.0, 'task_time_max_ms': 22.0, 'skew_ratio': 1.0, 'peak_mem_max_bytes': 262144.0}


,customer_id,orders,revenue
0,1430,7500,382700.18
1,1870,7500,382395.26
2,1970,7500,382243.44
3,1163,7500,382070.20
4,1630,7500,382013.80


## 3. Fix it (B) — `limit()` before collecting (the right way to peek)

If you just want to *look* at rows, bound how many cross to the driver. `df.limit(n).toPandas()`
ships at most `n` rows — the correct way to sample a huge frame. Then broadcast **only the small
side** (`dim`, not the huge fact): no large object transits the driver.

In [6]:
SAMPLE = 1_000
m_limit = measure(spark, "limit->pandas", lambda: wide_fact.limit(SAMPLE).toPandas())
print(f"limit({SAMPLE})->pandas metrics:", m_limit)

apply_profile(spark, "tuned")   # broadcast enabled (10 MB threshold)
good_broadcast = fact.join(F.broadcast(dim), on="customer_id", how="inner")   # small side only
m_bcast = measure(spark, "broadcast small dim", lambda: good_broadcast.count())
print("broadcast (small side) metrics:", m_bcast)

limit(1000)->pandas metrics: {'label': 'limit->pandas', 'runtime_s': 0.28, 'num_tasks': 9, 'shuffle_read_bytes': 0, 'shuffle_write_bytes': 3268716, 'spill_mem_bytes': 0, 'spill_disk_bytes': 0, 'task_time_median_ms': 14.0, 'task_time_max_ms': 18.0, 'skew_ratio': 1.3, 'peak_mem_max_bytes': 0.0}
Applied 'tuned' session profile:
  spark.sql.adaptive.enabled                       = true
  spark.sql.adaptive.skewJoin.enabled              = true
  spark.sql.adaptive.coalescePartitions.enabled    = true
  spark.sql.autoBroadcastJoinThreshold             = 10485760
  spark.sql.shuffle.partitions                     = 200
broadcast (small side) metrics: {'label': 'broadcast small dim', 'runtime_s': 0.24, 'num_tasks': 17, 'shuffle_read_bytes': 472, 'shuffle_write_bytes': 472, 'spill_mem_bytes': 0, 'spill_disk_bytes': 0, 'task_time_median_ms': 1.0, 'task_time_max_ms': 1.0, 'skew_ratio': 1.0, 'peak_mem_max_bytes': 0.0}


## 4. Prove it

Before/after. The broken approach never returns a result at all (it hits the `maxResultSize`
guardrail — see the captured error above); each fix returns a **small** result quickly because the
heavy work stayed on the cluster.

In [9]:
compare([m_broken, m_agg, m_limit, m_bcast])

print("\n'collect (huge, fails)' has no metrics: Spark blocked the >1 GB result before it could "
      "ship.\nThe fixes all return tiny results — big work stays on the cluster.")

| Metric              | collect (huge, fails) | aggregate->collect | limit->pandas | broadcast small dim |
| ------------------- | --------------------- | ------------------ | ------------- | ------------------- |
| Wall-clock runtime  |                     — |             0.44 s |        0.28 s |              0.24 s |
| Tasks               |                     — |                 10 |             9 |                  17 |
| Shuffle read        |                     — |           101.7 KB |           0 B |             472.0 B |
| Shuffle write       |                     — |           101.7 KB |        3.1 MB |             472.0 B |
| Spill (memory)      |                     — |                0 B |           0 B |                 0 B |
| Spill (disk)        |                     — |                0 B |           0 B |                 0 B |
| Task time — median  |                     — |              22 ms |         14 ms |                1 ms |
| Task time — max     |              

## Takeaways & "in real production…"

- **Never collect unbounded data to the driver.** `.collect()` / `.toPandas()` are for *small*,
  already-aggregated or `limit`ed results — treat them like `head()`, not an export.
- **Aggregate or write on the cluster**; bring back only what a human/dashboard can read.
- **Broadcast only genuinely small sides** — a broadcast of a large frame is a driver-OOM in
  disguise (it collects that side to the driver first).
- **The cap is a guardrail, not a fix.** `spark.driver.maxResultSize` turns a driver-heap death
  into a clean failure (and a smooth laptop), but the real fix is to stop moving big data to the
  driver. In prod: set the cap deliberately, alert on driver memory, and code-review for stray
  `.collect()` / `.toPandas()` on unbounded frames.

## Teardown

Nothing was written (the huge frame was never successfully collected; the fixes only return small
results), so there is nothing to delete. We restore the production-tuned safety nets.

In [10]:
apply_profile(spark, "tuned")        # restore production-tuned safety nets
spark.catalog.clearCache()
print("Done. Profile reset to 'tuned'. No tables/files were created; `make clean` clears .tmp if needed.")

SparkConnectException: [NO_ACTIVE_SESSION] No active Spark session found. Please create a new Spark session before running the code.